# Pandas Series

> 📘 **Python Mastery** · Module 11 — Pandas · Lesson 1/7

NumPy gave you fast arrays; **pandas** gives those arrays *labels*. A `Series` is the atom of pandas — one labeled column of data — and mastering it makes every later lesson (DataFrames, cleaning, groupby) feel natural.

## 🎯 Learning Objectives

- **Explain** what pandas is, what problem it solves over plain NumPy arrays, and how it relates to NumPy.
- **Create** a `pd.Series` from a list, from a dictionary, and with an explicit custom index.
- **Select** values safely with `.iloc` (position) and `.loc` / `[...]` (label), and know when to use which.
- **Inspect** a Series using its attributes: `index`, `values`, `dtype`, `name`, `shape`, `size`.
- **Predict** the result of vectorized arithmetic between two Series whose labels do not match.
- **Filter** a Series with a boolean mask and detect missing values with `isna()` / `notna()`.

## 1. What Is pandas, and Why Bother?

Real data rarely comes as bare numbers. It comes as *tables*: rows of students, columns named `age`, `city`, `score`. **pandas** is the Python library for exactly that — tabular data where every row and every column has a meaningful **label**, not just a position.

Think of it this way: a NumPy array is a smart shelf where you find things by slot number (`arr[3]`). A pandas object is a **filing cabinet**: every drawer has a written label, so you can ask for `"score"` instead of remembering "column 4".

Under the hood pandas is *built on NumPy* — you get NumPy's C speed plus labels, missing-data handling, group-by, joins, and file readers on top.

Install it once (in your terminal, not in a code cell):

```bash
pip install pandas
```

The whole world imports it with the nickname `pd`:

```python
import pandas as pd   # universal convention
import numpy as np    # pandas is built on numpy
```

In [ ]:
import pandas as pd   # convention: pandas answers to "pd"
import numpy as np

print(pd.__version__)   # any modern pandas 2.x / 3.x is fine for this course

## 2. Your First Series: From a List

A **`pd.Series`** is a one-dimensional labeled array — think *"one Excel column"* or *"one dictionary with superpowers"*. Build one by handing `pd.Series()` any list.

If you don't say otherwise, pandas attaches a default **`RangeIndex`**: the integers `0, 1, 2, ...`, one label per row.

**Syntax:**

```python
pd.Series(data, index=None, dtype=None, name=None)
# data     : list, tuple, dict, numpy array, scalar...
# index    : optional labels (must match len(data))
# name     : optional column-like name (useful later in DataFrames)
```

In [ ]:
import pandas as pd

scores = pd.Series([88, 92, 79, 95], name="final_score")
print(scores)          # left column = the index labels, right = the values
print()
print(type(scores))    # <class 'pandas.core.series.Series'>
print("shape:", scores.shape, "| size:", scores.size)   # (4,) and 4 -- one dimension

## 3. From a Dictionary: Labels for Free

Hand `pd.Series()` a dictionary and pandas uses the **keys as index labels** and the values as data. This is usually nicer than a RangeIndex — `"Dhaka"` means more than `0`.

You can also pass a list plus an explicit `index=` list to choose labels yourself. The lengths must match exactly, otherwise pandas raises a `ValueError`.

**Syntax:**

```python
pd.Series({"key1": v1, "key2": v2})          # keys become the labels
pd.Series([v1, v2], index=["label1", "label2"])  # labels chosen manually
```

In [ ]:
import pandas as pd

# dict -> keys become the index labels
population = pd.Series(
    {"Dhaka": 22.5, "Chattogram": 5.4, "Khulna": 2.0, "Sylhet": 3.9},
    name="population_millions",
)
print(population)
print()
print(list(population.index))   # the labels, as a plain list

In [ ]:
import pandas as pd

# explicit index: same values, our own labels
week_sales = pd.Series([4200, 5100, 3800],
                       index=["Sat", "Sun", "Mon"],
                       name="sales_taka")

print(week_sales)

# week_sales = pd.Series([4200, 5100, 3800], index=["Sat", "Sun"])
# ^ uncommenting this raises ValueError: Length of values (3) does not match length of index (2)

## 4. Getting Values Out: Position vs Label

There are two ways to point at a value, and mixing them up is the #1 beginner bug:

- **`.iloc[]`** — *"i"* for integer: select by **position**, counting from 0. Works on every Series.
- **`.loc[]`** — select by **label** (or `s["label"]`, which is shorthand for `.loc`).

⚠️ **House rule for this course:** *always write `.iloc` or `.loc` explicitly.* Bare `s[0]` is ambiguous — is `0` a position or a label? When the index itself contains integers, even pandas can get confused; explicit accessors never are.

**Syntax:**

```python
s.iloc[i]        # i-th value by POSITION      (like a list)
s.iloc[-1]       # negative positions work here
s.loc[label]     # value whose LABEL is given  (like a dict)
s["label"]       # same thing, shorter form
```

In [ ]:
import pandas as pd

temps = pd.Series([31.2, 33.8, 30.5, 29.9],
                  index=["Sat", "Sun", "Mon", "Tue"],
                  name="temp_c")

print(temps.iloc[0])    # FIRST position      -> 31.2
print(temps.iloc[-1])   # LAST position       -> 29.9
print(temps.loc["Mon"]) # value LABELED Mon   -> 30.5
print(temps["Tue"])     # shorthand for .loc  -> 29.9
print()
print(temps.iloc[1:3])  # slices work too: positions 1 and 2 (stop EXCLUSIVE, like lists)

In [ ]:
import pandas as pd

# The trap that justifies the house rule: an INTEGER-labeled Series
lottery = pd.Series(["toy", "car", "trip"], index=[10, 20, 30])

print(lottery.loc[10])    # label 10 -> "toy"
print(lottery.iloc[0])    # position 0 -> "toy" (same only by luck!)
print(lottery.iloc[1])    # position 1 -> "car", NOT label 20

# lottery[1]   <- deprecated territory: is 1 a label or a position?
# On integer indexes this ambiguity caused real bugs, so we never use bare [].
try:
    lottery.loc["a"]
except KeyError as e:
    print("KeyError:", e, "-> .loc looks for LABELS, .iloc looks for POSITIONS")

## 5. Series Attributes: Look Before You Compute

A Series carries metadata about itself. These **attributes** (no parentheses — they are properties, not methods) tell you what you are holding before you compute anything.

| Attribute | What it is |
|---|---|
| `index` | the labels |
| `values` | the data as a NumPy array |
| `dtype` | the data type of the values |
| `name` | the Series' own name (becomes a column name later) |
| `shape` / `size` | dimensions `(n,)` and element count |

**Example:**

In [ ]:
import pandas as pd
import numpy as np

stock = pd.Series({"pen": 120, "book": 45, "bag": 18}, name="units_in_stock")

print(stock.index)     # Index(['pen', 'book', 'bag'], dtype='str')
print(stock.values)    # plain numpy array lives underneath
print(stock.dtype)     # int64
print(stock.name)      # 'units_in_stock'
print(stock.shape, stock.size)

print(np.sqrt(stock.values[:2]))   # numpy functions accept the underlying array

## 6. Vectorized Math — and the Alignment Surprise

Because the data sits in NumPy arrays, arithmetic is **vectorized**: `series * 2` doubles every element with no loop, no list comprehension. That speed is half of why pandas exists.

The other half is **alignment**. When you combine *two* Series, pandas matches values **by label, not by position**. Labels found in only one operand produce `NaN` (missing) in the result — pandas refuses to guess what should pair with what.

> 🔍 **Under the Hood:** A Series wraps a NumPy array plus an `Index` object. Arithmetic like `a + b` first computes the **union** of both indexes, reorders each array so identical labels line up, then runs one C-level NumPy operation. Unmatched slots get filled with `NaN` — and because `NaN` is a float, an int+int result silently becomes `float64`. That dtype flip is your clue that missing values appeared.

In [ ]:
import pandas as pd

prices_bdt = pd.Series([150, 80, 1200], index=["book", "pen", "bag"], name="price")
doubled = prices_bdt * 2            # vectorized: every element doubled
print(doubled)
print()

discounted = prices_bdt - pd.Series([50, 50, 50], index=["book", "pen", "bag"])
print(discounted)                   # labels matched perfectly -> clean result

In [ ]:
import pandas as pd

mon = pd.Series([100, 200, 300], index=["Dhaka", "Sylhet", "Khulna"])
tue = pd.Series([10, 20],         index=["Sylhet", "Barishal"])

total = mon + tue                   # aligned BY LABEL
print(total)
print(total.dtype)                  # float64 -- ints were upcast because of NaN

# Dhaka appears only in `mon`, Barishal only in `tue` -> NaN there.
# Sylhet matched: 200 + 10 = 210.

## 7. Filtering with Boolean Masks

Ask the Series a True/False question and it answers with a boolean Series; feed that answer back in with square brackets and you keep only the `True` rows. This *mask → filter* pattern is used constantly — for DataFrames too.

**Syntax:**

```python
mask = s > value          # boolean Series, same labels as s
s[mask]                   # keep only the True positions
s[s > value]              # the same thing, one line
```

In [ ]:
import pandas as pd

scores = pd.Series([88, 92, 79, 95, 61], index=["Sarah", "Rafi", "Nabila", "Karim", "Mim"])

passed = scores >= 80              # step 1: the question
print(passed)                      # step 2: inspect the answer
print()
print(scores[passed])              # step 3: filter
print()
print(scores[scores > 90])         # all in one line: who scored above 90?
print()
print(scores.nsmallest(2))         # bonus: bottom-2 without sorting yourself

## 8. Missing Values: `isna()` and `notna()`

Real datasets have holes. pandas writes holes as `NaN` ("not a number"). Two sibling methods flag them:

- `s.isna()` → `True` where a value is **missing**
- `s.notna()` → `True` where a value is **present**

Both return boolean masks, so they compose beautifully with filtering. (Lesson 05 is dedicated to *fixing* these holes.)

**Example:**

In [ ]:
import pandas as pd
import numpy as np

visits = pd.Series([3, np.nan, 7, None, 2], index=["Mon", "Tue", "Wed", "Thu", "Fri"])
# np.nan and None BOTH count as missing in numeric Series

print(visits.isna())               # where are the holes?
print()
print("missing:", visits.isna().sum(), "| present:", visits.notna().sum())
print()
print(visits[visits.notna()])      # keep only real observations
print("mean ignoring NaN:", visits.mean())   # pandas skips NaN automatically

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Accessing with bare `s[0]` | Ambiguous: position or label? Breaks on integer indexes | Always write `s.iloc[0]` (position) or `s.loc[0]` (label) |
| Assuming `a + b` pairs by position | pandas aligns by **label**; mismatched labels become `NaN` | Check `a.index.equals(b.index)` or reindex first if you truly want positional math |
| Forgetting parentheses around masks | `s["a"] == 1 & s["b"] == 2` raises `TypeError` (`&` binds tighter than `==`) | Write `(s["a"] == 1) & (s["b"] == 2)` — full treatment in Lesson 06 |
| Expecting `int` Series to stay `int` after `NaN` appears | Any `NaN` forces `float64` | Accept floats, or handle missing values explicitly (`dropna`/`fillna`) |
| Trusting duplicate labels | `s["a"]` returns a whole Series, not a scalar, if `"a"` occurs twice | Keep index labels unique, or use `.iloc` for positions |

## 💡 Best Practices & Pro Tips

- **Name your Series** (`name="price"`): when it becomes a DataFrame column later, the name travels with it and saves you a rename.
- **Prefer vectorized operations** (`s * 2`, `s > x`) over Python loops — the difference is 10–100x on real data, and the code reads like the math.
- **Print `.index` and `.dtype` before computing.** Most "wrong answer" bugs in pandas are actually alignment or dtype surprises.
- Treat `.values` as read-only: it may be a view sharing memory with the Series. Use `.to_numpy(copy=True)` when you need an independent array.
- 🤖 **AI-engineering relevance:** feature engineering is Series arithmetic — normalizing a price column, masking outlier sensor readings, counting tokens per document. Every ML pipeline starts life as a handful of well-behaved Series.

## 📌 Summary

| Method / Attribute | What it does | Example |
|---|---|---|
| `pd.Series(list_or_dict)` | Create a labeled 1-D array | `pd.Series([88, 92], name="score")` |
| `.iloc[i]` | Select by position | `s.iloc[0]`, `s.iloc[-1]` |
| `.loc[label]` / `s[label]` | Select by label | `s.loc["Dhaka"]` |
| `.index`, `.values` | Labels, underlying NumPy array | `list(s.index)` |
| `.dtype`, `.shape`, `.size` | Type and dimensions | `s.dtype` |
| `s * 2`, `a + b` | Vectorized math; binary ops align on labels | `prices * 1.15` |
| `s[mask]` | Boolean filter | `s[s > 80]` |
| `isna()` / `notna()` | Locate missing / present values | `s.isna().sum()` |

**Key takeaways**

- A Series = NumPy values + labels; the labels are the whole point of pandas.
- `.iloc` = position, `.loc` = label. Write both explicitly, always.
- Binary operations align by label; unmatched labels become `NaN` and flip ints to floats.
- Masks in, filtered Series out — the core pattern behind all data selection.

## 🔗 Next Lesson

Next up: **[02_DataFrames](../02_DataFrames/notes.ipynb)** — stack many Series side by side and you get the pandas DataFrame, the workhorse table of data science.